# 01 — A minimal PINN from scratch**Making PINNs Work** · Prof. Dr. Dmitry MikhaylovEverything here is plain PyTorch. No PINN library, no hidden machinery — the point ofthis module is that you see every line that makes a physics-informed network work.We solve the viscous Burgers equation, the standard first benchmark in this field:$$u_t + u\,u_x - \nu\,u_{xx} = 0, \qquad x \in [-1, 1],\; t \in [0, 1]$$$$u(0, x) = -\sin(\pi x), \qquad u(t, -1) = u(t, 1) = 0, \qquad \nu = 0.01/\pi$$The solution starts as a smooth sine and steepens into a near-discontinuity at $x=0$.That steepening is what makes it a useful first test: a method that cannot resolve itwill look fine on the loss curve and wrong in the picture.Reference for the formulation: Raissi, Perdikaris & Karniadakis, *Journal ofComputational Physics* **378** (2019) 686–707.

In [ ]:
"""Module 01 - A minimal PINN from scratch: Burgers' equation.u_t + u*u_x - nu*u_xx = 0,  x in [-1,1], t in [0,1]u(0,x) = -sin(pi x),  u(t,-1) = u(t,1) = 0,  nu = 0.01/pi"""import timeimport numpy as npimport torchimport torch.nn as nntorch.manual_seed(0)np.random.seed(0)NU = 0.01 / np.pi

## The networkAn ordinary fully connected network taking $(t, x)$ and returning $u$. Nothing aboutthe architecture knows any physics — the physics lives entirely in the loss.`tanh` is used rather than `ReLU` for a concrete reason: we need second derivatives,and the second derivative of `ReLU` is zero almost everywhere.

In [ ]:
class MLP(nn.Module):    def __init__(self, width=64, depth=4):        super().__init__()        layers, d_in = [], 2        for _ in range(depth):            layers += [nn.Linear(d_in, width), nn.Tanh()]            d_in = width        layers += [nn.Linear(width, 1)]        self.net = nn.Sequential(*layers)        for m in self.net:            if isinstance(m, nn.Linear):                nn.init.xavier_normal_(m.weight)                nn.init.zeros_(m.bias)    def forward(self, t, x):        return self.net(torch.cat([t, x], dim=1))

## The physicsThis is the heart of the method. We need $u_t$, $u_x$ and $u_{xx}$ — and we get themfrom automatic differentiation of the network with respect to its **inputs**, not itsweights. No finite differences, no mesh.`create_graph=True` matters: it keeps the derivative itself differentiable, which iswhat lets us backpropagate through the residual and take $u_{xx}$ from $u_x$.

In [ ]:
def pde_residual(model, t, x):    t = t.clone().requires_grad_(True)    x = x.clone().requires_grad_(True)    u = model(t, x)    g = lambda y, v: torch.autograd.grad(y, v, torch.ones_like(y), create_graph=True)[0]    u_t = g(u, t)    u_x = g(u, x)    u_xx = g(u_x, x)    return u_t + u * u_x - NU * u_xx

## Where to enforce itThree sets of points: interior *collocation* points where the equation must hold,points at $t=0$ for the initial condition, and points on $x=\pm 1$ for the boundary.The collocation points are sampled at random and never change during training here.Module 06 shows how much this single choice is costing us.

In [ ]:
def sample(n_f=10000, n_0=200, n_b=200):    t_f = torch.rand(n_f, 1)    x_f = torch.rand(n_f, 1) * 2 - 1    x_0 = torch.rand(n_0, 1) * 2 - 1    t_0 = torch.zeros_like(x_0)    u_0 = -torch.sin(np.pi * x_0)    t_b = torch.rand(n_b, 1)    x_b = torch.where(torch.rand(n_b, 1) < 0.5, -torch.ones(n_b, 1), torch.ones(n_b, 1))    return t_f, x_f, t_0, x_0, u_0, t_b, x_b

## TrainingThe loss is a plain sum of three mean-squared terms — equation, initial condition,boundary. **This is the naive choice**, and module 02 is devoted to why it is a badone in general. It happens to work on this problem.Two optimisers in sequence. Adam gets into the right basin; L-BFGS then converges.This pairing is standard in the PINN literature and is worth internalising early — onthis problem Adam alone plateaus around $1.4\times10^{-3}$ and L-BFGS takes it toroughly $3.5\times10^{-5}$, a factor of forty, in about a third of the wall time.Expect roughly two minutes on a laptop CPU.

In [ ]:
def train(iters=5000, lr=1e-3, log_every=1000):    model = MLP()    opt = torch.optim.Adam(model.parameters(), lr=lr)    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2000, gamma=0.5)    t_f, x_f, t_0, x_0, u_0, t_b, x_b = sample()    t0 = time.time()    for it in range(1, iters + 1):        opt.zero_grad()        l_f = pde_residual(model, t_f, x_f).pow(2).mean()        l_0 = (model(t_0, x_0) - u_0).pow(2).mean()        l_b = model(t_b, x_b).pow(2).mean()        loss = l_f + l_0 + l_b        loss.backward()        opt.step()        sched.step()        if it % log_every == 0 or it == 1:            print(f"{it:6d}  total {loss.item():.3e}  pde {l_f.item():.3e} "                  f"ic {l_0.item():.3e}  bc {l_b.item():.3e}  {time.time()-t0:5.1f}s",                  flush=True)    # Adam gets close; L-BFGS is what actually converges a PINN.    print("\nL-BFGS refinement", flush=True)    lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=2000, history_size=50,                              tolerance_grad=1e-11, tolerance_change=1e-14,                              line_search_fn="strong_wolfe")    def closure():        lbfgs.zero_grad()        l = (pde_residual(model, t_f, x_f).pow(2).mean()             + (model(t_0, x_0) - u_0).pow(2).mean()             + model(t_b, x_b).pow(2).mean())        l.backward()        return l    lbfgs.step(closure)    with torch.no_grad():        after = ((model(t_0, x_0) - u_0).pow(2).mean() + model(t_b, x_b).pow(2).mean()).item()    after += pde_residual(model, t_f, x_f).pow(2).mean().item()    print(f"       total {after:.3e}  {time.time()-t0:5.1f}s", flush=True)    return model

## Checking the answerA low training loss is not evidence of a correct solution: the residual is onlysmall *at the points we trained on*. So we re-evaluate on twenty thousand freshpoints, check the initial condition directly, and confirm the shock actually formed.Module 08 turns this into a proper protocol. For now, treat it as the minimum.

In [ ]:
def validate(model):    """Low loss is not correctness. Check on points never trained on."""    tt = torch.rand(20000, 1)    xx = torch.rand(20000, 1) * 2 - 1    r = pde_residual(model, tt, xx).detach().abs()    x = torch.linspace(-1, 1, 1001).reshape(-1, 1)    t1 = torch.ones_like(x)    u1 = model(t1, x).detach().flatten()    grad = (u1[1:] - u1[:-1]).abs().max().item() / (x[1] - x[0]).item()    x0 = torch.linspace(-1, 1, 500).reshape(-1, 1)    ic_err = (model(torch.zeros_like(x0), x0).detach() + torch.sin(np.pi * x0)).abs().max().item()    print(f"\nfresh-grid residual  mean {r.mean():.3e}   max {r.max():.3e}")    print(f"initial condition    max abs error {ic_err:.3e}")    print(f"max |du/dx| at t=1   {grad:.1f}   (shock forms near x=0)")    print(f"solution range       [{u1.min():.3f}, {u1.max():.3f}]  (should stay within [-1,1])")

In [ ]:
model = train()validate(model)

## What you should see```  5000  total 1.444e-03  ...L-BFGS refinement       total 3.515e-05fresh-grid residual  mean 3.036e-03   max 1.480e-01initial condition    max abs error 7.983e-03max |du/dx| at t=1   82.8   (shock forms near x=0)solution range       [-0.711, 0.714]```Note the gap between the mean residual and the max residual — a factor of aboutfifty. The error is not spread evenly; it is concentrated at the shock, exactlywhere the solution is hardest. A single averaged number would have hidden that.## Plot itNumbers are not enough. Look at the solution.

In [ ]:
import matplotlib.pyplot as pltx = torch.linspace(-1, 1, 400).reshape(-1, 1)fig, ax = plt.subplots(figsize=(7, 4))for t_val in (0.0, 0.25, 0.50, 0.75, 1.0):    t = torch.full_like(x, t_val)    with torch.no_grad():        u = model(t, x)    ax.plot(x, u, label=f"t = {t_val:.2f}")ax.set_xlabel("x"); ax.set_ylabel("u"); ax.legend(); ax.grid(alpha=.3)ax.set_title("Burgers equation: a sine steepening into a shock")plt.tight_layout(); plt.show()

## Exercises1. Set `NU = 0.01/np.pi * 10`. The shock softens and everything converges faster.   Now go the other way, dividing by ten, and watch the method struggle.2. Replace `nn.Tanh` with `nn.ReLU` and explain the failure before you run it.3. Drop `n_f` to 500. How does the fresh-grid residual change relative to the   training loss? This is the gap module 08 is about.4. Remove the L-BFGS stage and give Adam 20 000 iterations instead. Compare final   accuracy and wall time.---Next: **02 — Why your loss will not go down**, where the naive equal weighting usedhere produces a 42% error on a problem it appears to be solving.